# Visualizacion del AST (Abstract Syntax Tree)

**Modulo:** Exploratory  
**Objetivo:** Comprender la estructura jerarquica del AST  
**Duracion estimada:** 25 minutos

---

## Contenido

1. [Setup](#setup)
2. [Estructura del AST](#estructura-del-ast)
3. [Jerarquia de Nodos](#jerarquia-de-nodos)
4. [Navegacion del AST](#navegacion-del-ast)
5. [Representacion en Diccionario](#representacion-en-diccionario)
6. [Ejemplos Practicos](#ejemplos-practicos)
7. [Ejercicios](#ejercicios)

---

## 1. Setup

In [ ]:
# Agregar path del proyecto
import sys
import json
sys.path.insert(0, '../..')

# Imports necesarios
from app.core.parser import parse_pseudocode
from app.core.parser.ast_nodes import (
    ProgramNode, AlgorithmNode, BlockNode,
    ForLoopNode, WhileLoopNode, IfStatementNode,
    AssignmentNode, ReturnStatementNode
)

print("Setup completado")

---

## 2. Estructura del AST

El Abstract Syntax Tree es una representacion jerarquica del codigo pseudocodigo.
Cada nodo del arbol representa una construccion sintactica del lenguaje.

### Nodo Raiz: ProgramNode

Todo AST comienza con un `ProgramNode` que contiene:

```
ProgramNode
|
+-- classes: List[ClassDefinitionNode]  # Definiciones de clases (opcional)
|
+-- algorithm: AlgorithmNode            # Algoritmo principal
```

### Nodo de Algoritmo: AlgorithmNode

Representa la definicion del algoritmo:

```
AlgorithmNode
|
+-- name: str                           # Nombre del algoritmo
|
+-- parameters: List[ParameterNode]     # Lista de parametros
|
+-- body: BlockNode                     # Cuerpo del algoritmo
```

### Ejemplo: Explorar la Estructura Basica

In [ ]:
# Ejemplo basico: estructura de un AST
codigo_simple = """
algorithm suma(n)
begin
    total <- 0
    return total
end
"""

ast = parse_pseudocode(codigo_simple)

# Explorar la estructura
print("=== ESTRUCTURA DEL AST ===")
print(f"Tipo raiz: {type(ast).__name__}")
print(f"Nombre algoritmo: {ast.algorithm.name}")
print(f"Numero de parametros: {len(ast.algorithm.parameters)}")
print(f"Parametros: {[p.name for p in ast.algorithm.parameters]}")
print(f"Numero de statements: {len(ast.algorithm.body.statements)}")

---

## 3. Jerarquia de Nodos

### Diagrama de Herencia

Todos los nodos heredan de la clase base abstracta `ASTNode`:

```
ASTNode (abstracta)
|
+-- ProgramNode
+-- AlgorithmNode
+-- ParameterNode
+-- BlockNode
+-- Statements
|   +-- AssignmentNode
|   +-- ForLoopNode
|   +-- WhileLoopNode
|   +-- IfStatementNode
|   +-- ReturnStatementNode
+-- Expressions
    +-- LiteralNode
    +-- VariableNode
    +-- BinaryOpNode
    +-- ArrayAccessNode
```

### Atributos Comunes

| Atributo | Tipo | Descripcion |
|----------|------|-------------|
| `node_type` | `ASTNodeType` | Tipo del nodo |
| `line` | `Optional[int]` | Linea en el codigo |
| `column` | `Optional[int]` | Columna en el codigo |

### Ejemplo: Inspeccionar Tipos de Nodos

In [ ]:
# Inspeccionar tipos de nodos en un algoritmo con ciclo
codigo_con_ciclo = """
algorithm buscar(A[], n, key)
begin
    for i <- 1 to n do
        if (A[i] = key) then
            return i
        end
    end
    return -1
end
"""

ast = parse_pseudocode(codigo_con_ciclo)

print("=== TIPOS DE NODOS ===")
for i, stmt in enumerate(ast.algorithm.body.statements):
    print(f"Statement {i+1}: {type(stmt).__name__}")
    
    # Si es un ForLoop, explorar su contenido
    if isinstance(stmt, ForLoopNode):
        print(f"  - Variable: {stmt.variable}")
        print(f"  - Body statements: {len(stmt.body.statements)}")

---

## 4. Navegacion del AST

### Estructura de un Ciclo For

```
ForLoopNode
|
+-- variable: str        # Variable de iteracion
+-- start: ExpressionNode # Valor inicial
+-- end: ExpressionNode   # Valor final
+-- body: BlockNode       # Cuerpo del ciclo
```

### Estructura de un Condicional If

```
IfStatementNode
|
+-- condition: ExpressionNode  # Condicion booleana
+-- then_block: BlockNode      # Bloque si verdadero
+-- else_block: BlockNode      # Bloque si falso (opcional)
```

### Ejemplo: Navegar Ciclos Anidados

In [ ]:
# Navegar ciclos anidados
codigo_anidado = """
algorithm bubbleSort(A[], n)
begin
    for i <- 1 to n - 1 do
        for j <- 1 to n - i do
            if (A[j] > A[j + 1]) then
                temp <- A[j]
                A[j] <- A[j + 1]
                A[j + 1] <- temp
            end
        end
    end
end
"""

ast = parse_pseudocode(codigo_anidado)

def contar_profundidad(node, nivel=0):
    """Funcion recursiva para contar profundidad de anidamiento"""
    max_nivel = nivel
    
    if hasattr(node, 'body') and node.body:
        for stmt in node.body.statements:
            max_nivel = max(max_nivel, contar_profundidad(stmt, nivel + 1))
    
    if hasattr(node, 'then_block') and node.then_block:
        for stmt in node.then_block.statements:
            max_nivel = max(max_nivel, contar_profundidad(stmt, nivel + 1))
    
    return max_nivel

profundidad = contar_profundidad(ast.algorithm)
print(f"Algoritmo: {ast.algorithm.name}")
print(f"Profundidad maxima de anidamiento: {profundidad}")

---

## 5. Representacion en Diccionario

Cada nodo del AST puede convertirse a diccionario usando el metodo `to_dict()`.
Esto es util para:

- Serializacion JSON
- Depuracion
- Exportacion

### Ejemplo: Convertir AST a JSON

In [ ]:
# Convertir AST a diccionario/JSON
codigo_simple = """
algorithm suma(n)
begin
    total <- 0
    for i <- 1 to n do
        total <- total + i
    end
    return total
end
"""

ast = parse_pseudocode(codigo_simple)

# Convertir a diccionario
ast_dict = ast.to_dict()

print("=== AST COMO DICCIONARIO ===")
print(json.dumps(ast_dict, indent=2)[:1000])  # Primeros 1000 caracteres
print("...")

---

## 6. Ejemplos Practicos

### Funcion de Visualizacion del AST

In [ ]:
# Funcion de visualizacion en formato arbol
def visualize_ast(node, indent=0):
    """Visualiza el AST en formato de arbol"""
    prefix = "  " * indent + ("|-- " if indent > 0 else "")
    node_name = type(node).__name__
    
    # Agregar info adicional segun el tipo
    info = ""
    if hasattr(node, 'name') and node.name:
        info = f" [{node.name}]"
    elif hasattr(node, 'variable'):
        info = f" [{node.variable}]"
    elif hasattr(node, 'value') and not hasattr(node, 'body'):
        info = f" = {node.value}"
    
    print(f"{prefix}{node_name}{info}")
    
    # Navegar hijos
    if hasattr(node, 'algorithm'):
        visualize_ast(node.algorithm, indent + 1)
    if hasattr(node, 'body') and node.body:
        if hasattr(node.body, 'statements'):
            for stmt in node.body.statements:
                visualize_ast(stmt, indent + 1)
    if hasattr(node, 'then_block'):
        for stmt in node.then_block.statements:
            visualize_ast(stmt, indent + 1)

# Visualizar Merge Sort
codigo_merge = """
algorithm mergeSort(A[], p, r)
begin
    if (p < r) then
        q <- floor((p + r) / 2)
        call mergeSort(A, p, q)
        call mergeSort(A, q + 1, r)
    end
end
"""

ast = parse_pseudocode(codigo_merge)
print("=== VISUALIZACION DEL AST ===")
visualize_ast(ast)

### Ejemplo: Comparar Estructuras de Algoritmos

In [ ]:
# Comparar estructuras de diferentes algoritmos
def analizar_estructura(codigo, nombre):
    """Analiza y resume la estructura de un algoritmo"""
    ast = parse_pseudocode(codigo)
    
    stats = {
        "nombre": nombre,
        "parametros": len(ast.algorithm.parameters),
        "statements": len(ast.algorithm.body.statements),
        "ciclos": 0,
        "condicionales": 0,
        "llamadas_recursivas": 0
    }
    
    def contar(node):
        if isinstance(node, ForLoopNode) or isinstance(node, WhileLoopNode):
            stats["ciclos"] += 1
        if isinstance(node, IfStatementNode):
            stats["condicionales"] += 1
        
        if hasattr(node, 'body') and node.body:
            for stmt in node.body.statements:
                contar(stmt)
        if hasattr(node, 'then_block'):
            for stmt in node.then_block.statements:
                contar(stmt)
    
    contar(ast.algorithm)
    return stats

# Analizar varios algoritmos
codigo_lineal = """
algorithm linearSearch(A[], n, key)
begin
    for i <- 1 to n do
        if (A[i] = key) then
            return i
        end
    end
    return -1
end
"""

codigo_bubble = """
algorithm bubbleSort(A[], n)
begin
    for i <- 1 to n - 1 do
        for j <- 1 to n - i do
            if (A[j] > A[j + 1]) then
                temp <- A[j]
            end
        end
    end
end
"""

print("=== COMPARACION DE ESTRUCTURAS ===")
for codigo, nombre in [(codigo_lineal, "Linear Search"), (codigo_bubble, "Bubble Sort")]:
    stats = analizar_estructura(codigo, nombre)
    print(f"\n{stats['nombre']}:")
    print(f"  Parametros: {stats['parametros']}")
    print(f"  Ciclos: {stats['ciclos']}")
    print(f"  Condicionales: {stats['condicionales']}")

---

## 7. Ejercicios

### Ejercicio 1: Contar Nodos por Tipo

In [ ]:
# Ejercicio 1: Implementa una funcion que cuente nodos por tipo
def contar_nodos_por_tipo(node, conteo=None):
    """Cuenta cuantos nodos hay de cada tipo en el AST"""
    if conteo is None:
        conteo = {}
    
    tipo = type(node).__name__
    conteo[tipo] = conteo.get(tipo, 0) + 1
    
    # Tu codigo aqui: navegar recursivamente todos los nodos hijos
    # Pista: revisar atributos como body, statements, then_block, etc.
    
    if hasattr(node, 'algorithm'):
        contar_nodos_por_tipo(node.algorithm, conteo)
    if hasattr(node, 'body') and node.body:
        contar_nodos_por_tipo(node.body, conteo)
    if hasattr(node, 'statements'):
        for stmt in node.statements:
            contar_nodos_por_tipo(stmt, conteo)
    
    return conteo

# Probar con un algoritmo
codigo_test = """
algorithm test(n)
begin
    for i <- 1 to n do
        if (i > 5) then
            x <- x + 1
        end
    end
end
"""

ast = parse_pseudocode(codigo_test)
conteo = contar_nodos_por_tipo(ast)

print("=== CONTEO DE NODOS ===")
for tipo, cantidad in sorted(conteo.items()):
    print(f"  {tipo}: {cantidad}")

---

## 8. Tips y Resumen

### Tips

- Usa `to_dict()` para depurar y exportar el AST
- Los nodos tienen atributos `line` y `column` para ubicacion en codigo
- Navega recursivamente para analizar estructuras anidadas
- Usa `isinstance()` para verificar tipos de nodos

### Resumen

Has aprendido:
- La estructura jerarquica del AST (ProgramNode, AlgorithmNode, BlockNode)
- Como navegar entre nodos usando atributos
- Como convertir el AST a formato JSON
- Como crear funciones de visualizacion recursivas
- Como analizar y comparar estructuras de algoritmos

---

## Proximos Pasos

- **grammar_experiments.ipynb**: Explorar la gramatica Lark en detalle
- **parser_testing.ipynb**: Probar parsing con diferentes sintaxis

**Siguiente modulo recomendado**: `02_complexity_analysis/` para analizar complejidad